# Byte Pair Encoding (BPE) Tokenizer from Scratch
## Friendly, beginner-first walkthrough

This notebook is a teaching version of the original BPE bonus notebook from *Build a Large Language Model (From Scratch)*.
It keeps the same code, but adds extra explanations for people who are new to tokenization and language-model preprocessing.

* Sec. 1, Sec. 3.


## Why this matters

Language models do not read words directly. They read **numbers** (token IDs).
A tokenizer is the translator from text to IDs and back again.

This notebook shows: how BPE works, how to train one tokenizer from scratch, how to encode and decode text, and how to load GPT-2-style vocab/merges.

Source material: [LLMs-from-scratch bonus BPE notebook](https://github.com/rasbt/LLMs-from-scratch).

* Sec. 1.


## How this differs from classic tokenization (pre-BPE era)

Before subword-BPE became common in NMT, many systems used simpler pipelines:

- **Word-level tokenization + fixed vocabulary**: map each known word to an ID and send unknown words to a single `<UNK>` token.
- **Back-off or dictionary tricks**: when a word is not in vocab, translation systems used copy rules, external dictionaries, or fallback logic.
- **Pure character models / fixed-length character n-grams**: they remove unknown words but often create very long sequences, which hurts context modeling and decoding efficiency.
- **Rule-based segmentation for specific languages**: compound splitters, hyphenation, or morpheme tools that are language- or linguistically-specific.

The key change in this paper is the **data-driven, language-agnostic merge process**:

- Start from basic symbols,
- repeatedly merge frequent adjacent pairs to form larger symbols,
- keep vocabulary size fixed by controlling merge count, and
- allow rare/unseen words to be represented by known subword pieces at inference time.

So instead of choosing between word-based and character-based trade-offs, BPE learns a **middle abstraction**: variable-length subwords.

This makes it much easier to balance:
1) **coverage** (open vocabulary behavior),
2) **efficiency** (fewer tokens than character level), and
3) **translation quality** on rare / unseen words.

* Sec. 3.1, Sec. 3.2, Sec. 4.1.


# 1. Main idea behind BPE

If you split text into **bytes**, each byte becomes its own token ID. This works, but it creates a lot of tokens.
BPE solves this by learning that some pieces appear together very often, and merging them into new tokens.

"Byte Pair" means exactly that: pairs of neighboring symbols are merged repeatedly.

* Sec. 3.2.


## 1.1 Bytes and characters (beginner intuition)

Think of text as a long string of symbols.
A tokenizer turns these symbols into IDs so a model can work with them.

### Step A: bytes from a sentence
`bytearray(..., 'utf-8')` shows the raw byte representation of the sentence.

We use this because the original GPT-family byte-level tokenizers build their roots from bytes.

* Sec. 1, Sec. 3.


In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)


bytearray(b'This is some text')


### Classic vs BPE comparison

#### Pre-BPE tokenizer (word-level baseline)


In [2]:
# Classic (pre-BPE) tokenizer: one ID per whitespace token
classic_words = text.split()
classic_word_to_id = {}
classic_ids = []
for tok in classic_words:
    if tok not in classic_word_to_id:
        classic_word_to_id[tok] = len(classic_word_to_id)
    classic_ids.append(classic_word_to_id[tok])

print('Classic word tokens:', classic_words)
print('Classic word IDs:', classic_ids)
print('Classic vocab size:', len(classic_word_to_id))


Classic word tokens: ['This', 'is', 'some', 'text']
Classic word IDs: [0, 1, 2, 3]
Classic vocab size: 4


### Step B: each byte is one integer
Each entry in that byte array is between 0 and 255.
That means a 1-byte-per-symbol approach can be very long for text, especially long inputs.

* Sec. 3.2.


In [3]:
ids = list(byte_ary)
print(ids)
print('Number of characters:', len(text))
print('Number of token IDs:', len(ids))


[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]
Number of characters: 17
Number of token IDs: 17


### Classic vs BPE comparison

#### Char-level pre-BPE baseline
Classic char tokenization still compresses differently from raw bytes.


In [4]:
# Classic character-level IDs (Unicode code points)
classic_char_ids = [ord(ch) for ch in text]
classic_chars = list(text)
print('Classic chars:', classic_chars)
print('Classic char IDs:', classic_char_ids)
print('Classic char token count:', len(classic_chars))
print('Byte vs char count ratio:', len(byte_ary) / max(1, len(classic_chars)))


Classic chars: ['T', 'h', 'i', 's', ' ', 'i', 's', ' ', 's', 'o', 'm', 'e', ' ', 't', 'e', 'x', 't']
Classic char IDs: [84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]
Classic char token count: 17
Byte vs char count ratio: 1.0


`17` characters became `17` IDs in this tiny example.
For larger text, this gets expensive very quickly.

BPE reduces this dramatically by learning common chunks like `This`, `text`, `is`, `some`, etc.
A model sees fewer IDs and can often process context more efficiently.


### 1.1.1 Why tokens can look weird in GPT-2

You may notice tokens like `Ġ` (special space marker) and strange newline glyphs (`Ċ`).
That is a convention used by GPT-2-style tokenizers; the same idea is still valid even if the token text representation is not plain-looking.

* Sec. 3.2.


# 1.2 Building a vocabulary

The vocabulary is the map \((\text{ID} \rightarrow \text{symbol})\). At the beginning, symbols are single bytes.
Each training step adds a new symbol made of two previous symbols that are merged together.

Over time, common fragments like `ent`, `ing`, `tion`, and even whole words can become single IDs.
This is why token counts shrink compared to raw bytes/chars.

* Sec. 3.2.


## 1.3 BPE algorithm outline (in plain English)

1. **Count**: find the most frequent adjacent pair.
2. **Merge**: replace every occurrence of that pair with a new symbol.
3. **Repeat**: keep doing this until budget is exhausted (vocabulary size reached) or no pair helps.
4. **Stop/Decode**: to decode, apply reverse rules in the right order.

A useful way to remember: this is like repeatedly learning a mini-dictionary of common chunks.

* Sec. 3.2.


## 1.3.1 Math & algorithm intuition

Let the current token sequence be

$$
x = (x_1, x_2, \dots, x_n)
$$

In one BPE step:

1) collect adjacent pairs

$$
P_i = (x_i, x_{i+1}),\; i = 1,\dots,n-1
$$

2) count each pair frequency

$$
f(p) = \text{count}(p)
$$

3) choose the best pair

$$
p^* = \arg\max_p f(p)
$$

4) replace non-overlapping occurrences of \(p^*\) with a new token id.

This is a greedy local optimization: each step uses the current pair frequencies only.

A simple compression check is

$$
\text{compression ratio} = \frac{|x|}{|\text{token\_ids}|}
$$

Higher ratio means fewer model input steps.

Special tokens such as \<|endoftext|\> are extra symbols added to the vocabulary map and merged with the same rule system.

* Sec. 3.2.


## 1.4 Concrete example

We encode the text `the cat in the hat` and watch it compress:

**Iteration 1**: merge `th` → `<256>`
`<256>e cat in <256>e hat`
**Iteration 2**: merge `<256>e` → `<257>`
`<257> cat in <257> hat`
**Iteration 3**: merge `<257> ` → `<258>`
`<258>cat in <258>hat`

Now decoding walks the chain backward to recover original text.

* Sec. 3.2 (Algorithm 1.)


# 2. A simple implementation (with beginner comments)

The class below is intentionally readable.
It does not aim for production speed; it aims for clarity.

You can use it in three ways:
- Train your own BPE tokenizer from text (`train`).
- Save/load your own vocab + merges.
- Load GPT-2 encoder files and mimic GPT-2-style tokenization behavior.

* Sec. 3.2.


In [5]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id -> token string (what each id means)
        self.vocab = {}
        # Reverse lookup: token string -> token id
        self.inverse_vocab = {}
        # Learned merges during training: (id1, id2) -> new_id
        self.bpe_merges = {}

        # GPT-2 style merge ranks: (symbol_A, symbol_B) -> rank
        # Lower rank = apply earlier / higher priority
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): Special tokens to include as-is.
        """

        # GPT-2-like preprocessing: use 'Ġ' as word-initial space marker
        # (space + following word becomes a single symbol chunk in many cases)
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Start vocab with base 256 byte values + seen training chars + space marker
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted(set(processed_text))
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add any requested special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Start from base token IDs for each processed symbol
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # Repeatedly merge the most frequent pair until target vocab size
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:
                break
            token_ids = self.replace_pair(token_ids, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Build merged token strings for every learned merge
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained GPT-2-compatible vocab and merge ranks.
        """
        # encoder.json stores token_str -> id
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}

        # Sanity checks for GPT-2-specific required entries
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph '\u012a' at id 198.")
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")

        # Keep newline alias for readable output
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Parse GPT-2 bpe merges into ranks (lower = higher priority)
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]

            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1

    def encode(self, text, allowed_special=None):
        """
        Encode text into token IDs.
        """
        # Special-token guard: avoid accidental unsupported control token text
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        token_ids = []

        # Handle allowed special tokens by splitting around them first
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
            text = text[last_index:]

            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        tokens = []
        # Newline handling is split out first so line breaks get stable treatment
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")
                        tokens.append("Ġ" + word)
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            for _ in range(pending_spaces):
                tokens.append("Ġ")

        # Finally map tokens to IDs, applying BPE if needed
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize one word-like segment using merge rules.
        """
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # Training mode (no GPT-2 ranks): merge only learned pairs in order
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        i += 2
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # GPT-2-style merge by rank
        symbols = [self.vocab[id_num] for id_num in token_ids]

        while True:
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            if bigram is None or bigram not in self.bpe_ranks:
                break

            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
            if len(symbols) == 1:
                break

        return [self.inverse_vocab[sym] for sym in symbols]

    def decode(self, token_ids):
        """Decode token IDs back to text."""
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """Save custom-trained vocab and merges to JSON."""
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """Load custom vocab and merges from JSON."""
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None
        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []
        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                dq.popleft()
            else:
                replaced.append(current)
        return replaced


### Classic vs BPE comparison

#### Minimal classic tokenization implementation
A fixed-word tokenizer with a small `<UNK>` fallback.


In [6]:
# Classic baseline: simple word-level tokenizer with unknown token
class WordTokenizerSimple:
    def __init__(self, vocab_size=256):
        self.vocab_size = vocab_size
        self.word_to_id = {}
        self.id_to_word = {}
        self.unk_id = 0
        self.word_to_id['<UNK>'] = self.unk_id
        self.id_to_word[self.unk_id] = '<UNK>'

    def train(self, text):
        for token in text.split():
            if token in self.word_to_id:
                continue
            if len(self.word_to_id) >= self.vocab_size:
                break
            new_id = len(self.word_to_id)
            self.word_to_id[token] = new_id
            self.id_to_word[new_id] = token

    def encode(self, text):
        return [self.word_to_id.get(token, self.unk_id) for token in text.split()]

    def decode(self, ids):
        return ' '.join(self.id_to_word.get(i, '<UNK>') for i in ids)

classic_demo = WordTokenizerSimple(vocab_size=100)
classic_demo.train(text)
classic_ids_demo = classic_demo.encode('This is some text')
classic_decoded_demo = classic_demo.decode(classic_ids_demo)
print('Classic IDs:', classic_ids_demo)
print('Classic decoded:', classic_decoded_demo)


Classic IDs: [1, 2, 3, 4]
Classic decoded: This is some text


### Python syntax quick notes

The tokenizer code uses several common patterns repeatedly:

- List comprehension: ` [f(x) for x in seq] `
- Dict lookup: `self.inverse_vocab[char]` and `dict.get(char)`
- File context manager:
  ```python
with open(path, 'r', encoding='utf-8') as file:
    ...
  ```
- f-string interpolation:
  ```python
raise ValueError(f'Disallowed special tokens: {disallowed}')
  ```
- `@lru_cache` memoizes pure function outputs (used for fast repeated lookups).
- Regex helpers from `re`:
  - `re.finditer(pattern, text)` finds matching spans.
  - `re.split(r'(\\r\\n|\\r|\\n)', text)` splits and keeps newline separators.

A good habit is to read each complex line as: call + arguments + assignment target.

Common indexing notation used in the notebook:

$$
x_i \;\; \text{means the } i\text{-th element of a sequence }x
$$

* Sec. 3.2.


### Quick sanity check: what we just built

- `train()` learns merges from text, storing rules.
- `encode()` applies these rules greedily to compress new text into IDs.
- `decode()` reverses back to readable text.

If `decode(encode(text))` matches the input text, your tokenizer is behaving consistently.

* Sec. 3.2.


## 3. Loading sample data

We use `the-verdict.txt` from the original repository as a toy training corpus.
The text is small, so the resulting merges will be limited compared to a real-world tokenizer.

* Sec. 4.


In [7]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path


verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f:
    text = f.read()


the-verdict.txt already exists in ../01_main-chapter-code/the-verdict.txt


### Classic vs BPE comparison

#### Data loading in classic pipelines
Classic systems usually work on whitespace tokens directly from text.


In [66]:
# Classic corpus summary (word-token view)
raw_tokens = text.split()
print('Classic corpus token count:', len(raw_tokens))
print('Classic unique token count:', len(set(raw_tokens)))
print('Classic first 20 tokens:', raw_tokens[:20])


Classic corpus token count: 3634
Classic unique token count: 1485
Classic first 20 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius--though', 'a', 'good', 'fellow', 'enough--so', 'it', 'was', 'no', 'great', 'surprise', 'to']


## 3.1 Train your own BPE tokenizer

We set a target vocab size of `1000` in this small example.
Base 256 byte tokens already exist, so this adds only a few hundred merges.

* Sec. 3.2.


In [67]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})
print("Vocab size:", len(tokenizer.vocab))
print("Number of merges:", len(tokenizer.bpe_merges))


Vocab size: 1000
Number of merges: 742


### Classic vs BPE comparison

#### Word-vocab training (fixed-size)
Classic approach: keep top-N words, map rest to `<UNK>`.


In [68]:
# Classic fixed-size word vocabulary
from collections import Counter
word_counts = Counter(text.split())
target_vocab = 1000
classic_top_words = [w for w, _ in word_counts.most_common(target_vocab - 1)]
classic_word_vocab = {'<UNK>': 0}
for w in classic_top_words:
    if w not in classic_word_vocab:
        classic_word_vocab[w] = len(classic_word_vocab)
print('Classic top words selected:', len(classic_top_words))
print('Classic word vocab size with <UNK>:', len(classic_word_vocab))
print('Is <UNK> present?', '<UNK>' in classic_word_vocab)


Classic top words selected: 999
Classic word vocab size with <UNK>: 1000
Is <UNK> present? True


Note: more training text and larger vocab generally produce longer subword chunks
instead of many tiny 2-character pieces.


In [69]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

# Try with a special token passed through
input_text_special = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids_special = tokenizer.encode(input_text_special, allowed_special={"<|endoftext|>"})
print(token_ids_special)
print("Number of chars:", len(input_text_special))
print("Number of tokens:", len(token_ids_special))


[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]
[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]
Number of chars: 56
Number of tokens: 22


### Classic vs BPE comparison

#### Encoding a sentence
Classic tokenizer encodes by word, BPE encodes by learned subword merges.


In [70]:
# Classic baseline encoding for the same text
classic_vocab_local = {'<UNK>': 0, '<|endoftext|>': 1}
for w in input_text.split():
    if w not in classic_vocab_local:
        classic_vocab_local[w] = len(classic_vocab_local)

classic_encoded = [classic_vocab_local[w] for w in input_text.split()]
print('Classic encoded:', classic_encoded)

classic_input_tokens = input_text_special.split()
classic_encoded_special = [
    classic_vocab_local.get(tok, classic_vocab_local['<UNK>'])
    for tok in classic_input_tokens
]
print('Classic with special token tokenization:', classic_encoded_special)


Classic encoded: [2, 3, 4, 5, 6, 7, 8]
Classic with special token tokenization: [2, 3, 4, 5, 6, 7, 0]


If the counts are lower than raw characters, tokenization compressed the text into meaningful chunks.


In [71]:
print(token_ids_special)
print(tokenizer.decode(token_ids_special))


[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]
Jack embraced beauty through art and life.<|endoftext|> 


### Classic vs BPE comparison

#### Decoding comparison
Classic decode is just ID -> word lookup; less expressive for rare strings.


In [72]:
# Classic baseline decode for the same sample
classic_id_to_word = {0: '<UNK>', 1: '<|endoftext|>'}
for tok in input_text_special.split():
    if tok not in classic_id_to_word.values():
        new_id = len(classic_id_to_word)
        classic_id_to_word[new_id] = tok

classic_tokens_back = [
    classic_id_to_word.get(i, '<UNK>')
    for i in classic_encoded_special
]
print('Classic decoded tokens:', classic_tokens_back)
print('Classic decoded string:', ' '.join(classic_tokens_back))


Classic decoded tokens: ['Jack', 'embraced', 'beauty', 'through', 'art', 'and', '<UNK>']
Classic decoded string: Jack embraced beauty through art and <UNK>


You can inspect one token at a time to see what each ID represents.


In [73]:
for token_id in token_ids_special:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")


424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .
257 -> <|endoftext|>
256 ->  


### Classic vs BPE comparison

#### Token-by-token inspection
Word-level tokenizer produces fewer internal boundaries than BPE subword IDs.


In [74]:
# Classic equivalent: inspect each word id and label
for i, token_id in enumerate(classic_encoded_special):
    print(token_id, '->', classic_id_to_word[token_id])

print('Number of classic IDs:', len(classic_encoded_special))
print('Number of BPE IDs:', len(token_ids_special))


2 -> Jack
3 -> embraced
4 -> beauty
5 -> through
6 -> art
7 -> and
0 -> <UNK>
Number of classic IDs: 7
Number of BPE IDs: 22


## 3.2 Round-trip check

If everything is wired correctly, tokenizing then detokenizing should give you back the same text.

* Sec. 3.2.


In [75]:
print(tokenizer.decode(tokenizer.encode("This is some text.")))
print(tokenizer.decode(tokenizer.encode("This is some text with \n newline characters.")))


This is some text.
This is some text with 
 newline characters.


### Classic vs BPE comparison

#### Round-trip check
Classic methods using simple split+join can lose formatting details.


In [76]:
# Classic round-trip (word-level):
classic_rt_text = 'This is some text'
classic_rt_ids = [classic_vocab_local.get(tok, 0) for tok in classic_rt_text.split()]
classic_rt_restored = ' '.join(classic_id_to_word.get(i, '<UNK>') for i in classic_rt_ids)
print(classic_rt_text == classic_rt_restored)
print('Classic restored:', classic_rt_restored)

classic_newline = 'This is some text with \n newline characters.'
classic_newline_ids = [classic_vocab_local.get(tok, 0) for tok in classic_newline.split()]
classic_newline_restored = ' '.join(classic_id_to_word.get(i, '<UNK>') for i in classic_newline_ids)
print(classic_newline_restored)


False
Classic restored: <UNK> <UNK> <UNK> <UNK>
<UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>


## 3.3 Save and load your trained tokenizer

Persistence is important: train once, reuse many times across sessions or deployment code.

* Sec. 3.2.


In [77]:
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")
print(tokenizer2.decode(token_ids_special))


Jack embraced beauty through art and life.<|endoftext|> 


### Classic vs BPE comparison

#### Persistence of a classic tokenizer
Classic tokenizers also persist dictionary artifacts (often JSON or binary blobs).


In [78]:
# Classic save/load vocabulary only
import json
classic_vocab_file = 'classic_word_vocab.json'
classic_word_vocab_simple = {'<UNK>': 0, 'This': 1, 'is': 2, 'some': 3, 'text': 4}
with open(classic_vocab_file, 'w', encoding='utf-8') as f:
    json.dump(classic_word_vocab_simple, f, ensure_ascii=False, indent=2)
with open(classic_vocab_file, 'r', encoding='utf-8') as f:
    classic_word_vocab_loaded = json.load(f)
print('Classic vocab loaded keys:', len(classic_word_vocab_loaded))


Classic vocab loaded keys: 5


## 3.4 Load GPT-2 tokenizer files

This version uses official OpenAI GPT-2 vocab + merges files so you can compare against known GPT-2 behavior.

* Sec. 3.2.


In [79]:
# Download GPT-2 files if missing
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)
print("GPT-2 vocab size:", len(tokenizer_gpt2.vocab))


vocab.bpe already exists in ch02/02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
encoder.json already exists in ch02/02_bonus_bytepair-encoder/gpt2_model/encoder.json
GPT-2 vocab size: 50257


### Classic vs BPE comparison

#### GPT-2 file loading step in a classic baseline
Classic systems usually skip external merge files and keep only word vocab.


In [80]:
# Classic alternative: create tokenizer from local words without merge files
_classic_local_words = sorted(set(text.split()))[:1000]
classic_local_vocab = {w: i for i, w in enumerate(_classic_local_words)}
classic_local_vocab['<UNK>'] = classic_local_vocab.get('<UNK>', len(classic_local_vocab))
print('Classic local vocab size:', len(classic_local_vocab))


Classic local vocab size: 1001


In [81]:
input_text = "This is some text"
gpt2_ids = tokenizer_gpt2.encode(input_text)
print(gpt2_ids)
print(tokenizer_gpt2.decode(gpt2_ids))


[1212, 318, 617, 2420]
This is some text


### Classic vs BPE comparison

#### GPT-2 encode step vs classic baseline
Classic: fixed word IDs, no subword merge hierarchy.


In [82]:
# Classic word-level encode of same input
classic_vocab_for_gpt2 = {
    '<UNK>': 0,
    'This': 1,
    'is': 2,
    'some': 3,
    'text': 4,
}
classic_gpt2_ids = [classic_vocab_for_gpt2.get(tok, 0) for tok in input_text.split()]
classic_gpt2_id_to_token = {i: tok for tok, i in classic_vocab_for_gpt2.items()}
classic_gpt2_text = ' '.join(classic_gpt2_id_to_token.get(i, '<UNK>') for i in classic_gpt2_ids)
print('Classic ids:', classic_gpt2_ids)
print('Classic reconstructed:', classic_gpt2_text)


Classic ids: [1, 2, 3, 4]
Classic reconstructed: This is some text


## Conclusion

You now have
- A mental model for BPE tokenization,
- A complete minimal BPE class implementation, and
- A practical path to train, persist, and load tokenizers.

For real projects, `tiktoken` is much faster and battle-tested.
This notebook is intentionally educational so you can understand each step.

* Sec. 5.


## 4. Beginner exercises

Try these three quick checks to test your intuition.

### Exercise 1: token length intuition
Take the same text and compare raw byte length vs BPE token length after training.

* Sec. 4.


In [8]:
text_small = "Tokenization makes language models efficient."
raw_byte_len = len(bytearray(text_small, "utf-8"))
print("Raw bytes:", raw_byte_len)

tokenizer_ex1 = BPETokenizerSimple()
tokenizer_ex1.train(text_small, vocab_size=400, allowed_special={"<|endoftext|>"})
bpe_ids = tokenizer_ex1.encode(text_small)
print("BPE token IDs:", len(bpe_ids))
print("BPE tokens:", bpe_ids)


Raw bytes: 45
BPE token IDs: 31
BPE tokens: [270, 259, 97, 258, 115, 256, 108, 97, 110, 103, 117, 97, 103, 101, 259, 111, 100, 101, 108, 115, 256, 101, 102, 102, 105, 99, 105, 101, 110, 116, 46]


### Classic vs BPE comparison

#### Token-length efficiency baseline
Classic baselines: word- and char-level counts.


In [9]:
classic_word_len_small = len(text_small.split())
classic_char_len_small = len(text_small)
print('Classic word-level length:', classic_word_len_small)
print('Classic character length:', classic_char_len_small)
print('BPE length (from previous cell):', len(bpe_ids))
print('BPE tokens per raw byte:', len(bpe_ids) / max(1, raw_byte_len))


Classic word-level length: 5
Classic character length: 45
BPE length (from previous cell): 31
BPE tokens per raw byte: 0.6888888888888889


### Exercise 2: observe how merges are learned
Train with two vocabulary sizes and compare how compressed each output gets.

* Sec. 4.1.


In [10]:
sample = "BPE learns frequent pairs step by step, like a human noticing patterns."

tok_300 = BPETokenizerSimple()
tok_300.train(sample, vocab_size=300, allowed_special={"<|endoftext|>"})
ids_300 = tok_300.encode(sample)
print("vocab=300 -> tokens:", len(ids_300))

tok_800 = BPETokenizerSimple()
tok_800.train(sample, vocab_size=800, allowed_special={"<|endoftext|>"})
ids_800 = tok_800.encode(sample)
print("vocab=800 -> tokens:", len(ids_800))

print("Difference:", len(ids_300) - len(ids_800), "fewer tokens with larger vocab")


vocab=300 -> tokens: 50
vocab=800 -> tokens: 50
Difference: 0 fewer tokens with larger vocab


### Classic vs BPE comparison

#### Merge count vs vocab-size behavior
In classic word models, vocabulary size is usually fixed by top-k words.


In [86]:
from collections import Counter
sample_words = sample.split()
sample_counts = Counter(sample_words)
for limit in [300, 800]:
    top_words = [w for w, _ in sample_counts.most_common(limit)]
    # Classic IDs per token if using top-k word vocab + <UNK>
    classic_vocab_limit = {'<UNK>': 0}
    for w in top_words:
        if w not in classic_vocab_limit:
            classic_vocab_limit[w] = len(classic_vocab_limit)
    classic_encoded_limit = [classic_vocab_limit.get(w, 0) for w in sample_words]
    print(f'classic top-{limit} vocab -> ids:', len(classic_encoded_limit))


classic top-300 vocab -> ids: 12
classic top-800 vocab -> ids: 12


### Exercise 3: decode round-trip with special token handling
Use a text that contains a special token and confirm encode -> decode works exactly.

* Sec. 3.2.


In [87]:
sample_with_special = "This is a test sentence.<|endoftext|> End."
ids = tokenizer.encode(sample_with_special, allowed_special={"<|endoftext|>"})
restored = tokenizer.decode(ids)
print("IDs:", ids)
print("restored == original?:", restored == sample_with_special)
print("restored text:", restored)


IDs: [542, 299, 256, 299, 287, 259, 315, 116, 321, 360, 271, 99, 101, 46, 257, 256, 69, 466, 46]
restored == original?: True
restored text: This is a test sentence.<|endoftext|> End.


### Classic vs BPE comparison

#### Special token handling
Classic tokenizers often keep special control tokens as literal extra vocabulary entries.


In [88]:
# Classic handling of special token marker
classic_vocab_with_special = {
    '<UNK>': 0,
    '<|endoftext|>': 1,
}
classic_word_list = sample_with_special.split()
for tok in classic_word_list:
    if tok not in classic_vocab_with_special:
        classic_vocab_with_special[tok] = len(classic_vocab_with_special)
classic_ids_special = [classic_vocab_with_special[t] for t in classic_word_list]
classic_decoded_special = ' '.join([tok if tok == '<|endoftext|>' else tok for tok in classic_word_list])
print('Classic special IDs:', classic_ids_special)
print('Classic restored contains marker:', '<|endoftext|>' in sample_with_special)


Classic special IDs: [2, 3, 4, 5, 6, 7]
Classic restored contains marker: True


## 5. Visual summary (Mermaid)

```mermaid
flowchart LR
    A["Raw input text"] --> B["Preprocess text\n(space / newline / specials)"]
    B --> C["Convert to initial symbols / IDs"]
    C --> D["Count frequent adjacent pairs"]
    D --> E["Create merge rule\n(pair -> new ID)"]
    E --> F["Repeat merge loop\nuntil vocab limit or no pair"]
    F --> G["Learned vocab + merges"]
    G --> H["Encode new text\n(apply merges)"]
    G --> I["Decode IDs\n(reverse merges)"]
    H --> J["Token IDs"]
    I --> K["Reconstructed text"]
    J --> I

```

* Sec. 3.2, Sec. 4.


## 6. Compact offline version (no external links)

If you want a quick local workflow, remember this sequence:
1. `tokenizer = BPETokenizerSimple()`
2. `tokenizer.train(train_text, vocab_size=...)`
3. `ids = tokenizer.encode(text)`
4. `recovered = tokenizer.decode(ids)`

That is the full life cycle: train once, then encode and decode anywhere.

* Sec. 3.2, Sec. 4.


## Appendix: concept glossary

- **byte**: values in the range \([0,255]\) representing text as numbers in UTF-8.
- **token**: one integer ID consumed by the model.
- **vocab**: map \((id \mapsto \text{text})\) and inverse \((\text{text} \mapsto id)\).
- **pair**: two adjacent symbols \((a,b)\).
- **pair frequency**: how often a pair appears next to each other.
- **merge**: replace \((a,b)\) with one new symbol id.
- **bpe\_merges**: learned rules \((id_1,id_2) \to id_{new}\).
- **special token**: reserved token like \(<|endoftext|>\).

Quick algebra:

If \(n\) original units become \(m\) token IDs after encoding:

$$
\text{compression ratio} = \frac{n}{m}
$$

Larger vocab sizes usually increase compression because more groups can be merged.

This notebook keeps code readable for study first, then performance tuning can be added separately.

* Sec. 3, Sec. 4.1.


In [ ]:
%%sql
